# Graph-conditioned prover (the full GMW-GI input)

We include the actual graphs `G0, G1` in the sequence, with
`G1 = φ(G0)`, and check that conditioning on the graphs does not
change the leak (the attack lives in the ψ-vs-φ structure).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root
import torch, itertools, math
torch.manual_seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


device: cuda


In [2]:
from subliminal.data import rand_graphs, rand_perms, apply_perm_to_graph, is_isomorphism
n = 5; g = torch.Generator().manual_seed(0)
g0 = rand_graphs(4, n, g); phi = rand_perms(4, n, g)
g1 = apply_perm_to_graph(g0, phi, n)
print('G1 = phi(G0) holds for all rows:', bool(is_isomorphism(phi, g0, g1, n).all()))


G1 = phi(G0) holds for all rows: True


The graph-conditioned sequence layout:


In [3]:
from subliminal.multi_graph import graph_multi_layout, graph_seq_len, build_graph_multi_example
MAX_N = 7
toks, labels = build_graph_multi_example(n, g0[0], phi[0], rand_perms(1, n, g)[0], MAX_N)
lay = graph_multi_layout(n, MAX_N)
print('sequence length (max):', graph_seq_len(MAX_N))
for name in ['g0','g1','phi','psi','psi_inv','phi_psi_inv']:
    print(f'  {name:12s} tokens:', toks[lay[name]].tolist())


sequence length (max): 76
  g0           tokens: [1, 0, 1, 1, 1, 0, 1, 0, 1, 0]
  g1           tokens: [0, 0, 1, 0, 1, 0, 1, 1, 1, 1]
  phi          tokens: [3, 4, 1, 0, 2]
  psi          tokens: [2, 3, 1, 0, 4]
  psi_inv      tokens: [3, 2, 0, 1, 4]
  phi_psi_inv  tokens: [0, 1, 3, 4, 2]


Training a full graph-conditioned shared model and running the
attack follows exactly the same recipe — see
`experiments/multi_graph.py`. The result: **with ≈ without**
`(G0,G1)` conditioning, so abstracting the graphs away does not
change the leak.
